In [ ]:
# Preprocessing and feature encoding
# Import necessary libraries
import gdown
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [2]:
# Google Drive file ID and download URL
# file_id = "1JEflC5W9gV0Z039OYCT2hOxE9G9KsqOj" #Madiha dataset
file_id = "1Chr4Gs06TDX-T81vwCNW6dP57JPhEX7x"  # Nooreen dataset

url = f"https://drive.google.com/uc?id={file_id}"
# Download the file
output = "cleaned_p2p_lending_data.csv"
gdown.download(url, output, quiet=False)

# Read the CSV into a DataFrame
df = pd.read_csv(output)
df.head()

Downloading...
From (original): https://drive.google.com/uc?id=1Chr4Gs06TDX-T81vwCNW6dP57JPhEX7x
From (redirected): https://drive.google.com/uc?id=1Chr4Gs06TDX-T81vwCNW6dP57JPhEX7x&confirm=t&uuid=69c945eb-be14-4f5b-991f-42460837c200
To: d:\GitHub\ET6-CDSP-group-15-repo\4_data_analysis\notebooks\cleaned_p2p_lending_data.csv
100%|██████████| 279M/279M [00:22<00:00, 12.2MB/s] 


,loan_amnt,term,int_rate,annual_inc,dti,delinq_2yrs,inq_last_6mths,emp_length,fico_range_low,fico_range_high,...,addr_state_WV,addr_state_WY,grade_A,grade_B,grade_C,grade_D,grade_E,grade_F,grade_G,credit_history_length
0,3600.0,36,13.99,55000.0,5.91,0.0,1.0,10,675.0,679.0,...,0,0,0,0,1,0,0,0,0,150.0
1,24700.0,36,11.99,65000.0,16.06,1.0,4.0,10,715.0,719.0,...,0,0,0,0,1,0,0,0,0,195.0
2,20000.0,60,10.78,63000.0,10.78,0.0,0.0,10,695.0,699.0,...,0,0,0,1,0,0,0,0,0,187.0
3,10400.0,60,22.45,104433.0,25.37,1.0,3.0,3,695.0,699.0,...,0,0,0,0,0,0,0,1,0,213.0
4,11950.0,36,13.44,34000.0,10.20,0.0,0.0,4,690.0,694.0,...,0,0,0,0,1,0,0,0,0,343.0


In [30]:
df.columns.to_list()

['loan_amnt',
 'term',
 'int_rate',
 'annual_inc',
 'dti',
 'delinq_2yrs',
 'inq_last_6mths',
 'emp_length',
 'fico_range_low',
 'fico_range_high',
 'open_acc',
 'pub_rec',
 'revol_bal',
 'revol_util',
 'total_acc',
 'application_type',
 'is_default',
 'is_verified',
 'home_MORTGAGE',
 'home_OWN',
 'home_RENT',
 'purpose_credit_card',
 'purpose_debt_consolidation',
 'purpose_home_improvement',
 'purpose_major_purchase',
 'purpose_other',
 'addr_state_AL',
 'addr_state_AR',
 'addr_state_AZ',
 'addr_state_CA',
 'addr_state_CO',
 'addr_state_CT',
 'addr_state_DC',
 'addr_state_DE',
 'addr_state_FL',
 'addr_state_GA',
 'addr_state_HI',
 'addr_state_IA',
 'addr_state_ID',
 'addr_state_IL',
 'addr_state_IN',
 'addr_state_KS',
 'addr_state_KY',
 'addr_state_LA',
 'addr_state_MA',
 'addr_state_MD',
 'addr_state_ME',
 'addr_state_MI',
 'addr_state_MN',
 'addr_state_MO',
 'addr_state_MS',
 'addr_state_MT',
 'addr_state_NC',
 'addr_state_ND',
 'addr_state_NE',
 'addr_state_NH',
 'addr_state_NJ',


In [16]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
# Preprocess Data
X = df.drop("is_default", axis=1)
y = df["is_default"]

# Scale numeric features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled, y = smote.fit_resample(X_scaled, y)

# Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

In [ ]:
# Train a Multi-Layer Perceptron (MLP) with 32 hidden layers (deep model)
model = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation="relu",
    solver="adam",
    learning_rate_init=0.0005,
    alpha=0.0001,  # L2 penalty (avoid too high)
    max_iter=500,
    random_state=42,
)


# Fit model
model.fit(X_train, y_train)

In [19]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

In [26]:
# # Make predictions
y_proba_40 = model.predict_proba(X_test)[:, 1]
y_pred_40 = (y_proba >= 0.4).astype(int)

In [27]:
# Evaluation function
def evaluate_model(y_test, y_pred, y_proba, model_name):
    print(f"\n=== {model_name} ===")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_proba))


evaluate_model(y_test, y_pred, y_proba, "MLP Classifier")
evaluate_model(y_test, y_pred_40, y_proba_40, "MLP Classifier with 40% threshold")


=== MLP Classifier ===
              precision    recall  f1-score   support

           0       0.74      0.84      0.79    203592
           1       0.82      0.71      0.76    203747

    accuracy                           0.78    407339
   macro avg       0.78      0.78      0.78    407339
weighted avg       0.78      0.78      0.78    407339

Confusion Matrix:
 [[171313  32279]
 [ 58687 145060]]
ROC-AUC: 0.862282958645219

=== MLP Classifier with 40% threshold ===
              precision    recall  f1-score   support

           0       0.78      0.73      0.76    203592
           1       0.75      0.80      0.77    203747

    accuracy                           0.76    407339
   macro avg       0.77      0.76      0.76    407339
weighted avg       0.77      0.76      0.76    407339

Confusion Matrix:
 [[148924  54668]
 [ 41366 162381]]
ROC-AUC: 0.862282958645219


In [33]:
from sklearn.inspection import permutation_importance

result = permutation_importance(
    model, X_test, y_test, scoring="f1", n_repeats=5, random_state=42
)

feature_names = [
    "loan_amnt",
    "term",
    "int_rate",
    "annual_inc",
    "dti",
    "delinq_2yrs",
    "inq_last_6mths",
    "emp_length",
    "fico_range_low",
    "fico_range_high",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "application_type",
    "is_verified",
    "home_MORTGAGE",
    "home_OWN",
    "home_RENT",
    "purpose_credit_card",
    "purpose_debt_consolidation",
    "purpose_home_improvement",
    "purpose_major_purchase",
    "purpose_other",
    "addr_state_AL",
    "addr_state_AR",
    "addr_state_AZ",
    "addr_state_CA",
    "addr_state_CO",
    "addr_state_CT",
    "addr_state_DC",
    "addr_state_DE",
    "addr_state_FL",
    "addr_state_GA",
    "addr_state_HI",
    "addr_state_IA",
    "addr_state_ID",
    "addr_state_IL",
    "addr_state_IN",
    "addr_state_KS",
    "addr_state_KY",
    "addr_state_LA",
    "addr_state_MA",
    "addr_state_MD",
    "addr_state_ME",
    "addr_state_MI",
    "addr_state_MN",
    "addr_state_MO",
    "addr_state_MS",
    "addr_state_MT",
    "addr_state_NC",
    "addr_state_ND",
    "addr_state_NE",
    "addr_state_NH",
    "addr_state_NJ",
    "addr_state_NM",
    "addr_state_NV",
    "addr_state_NY",
    "addr_state_OH",
    "addr_state_OK",
    "addr_state_OR",
    "addr_state_PA",
    "addr_state_RI",
    "addr_state_SC",
    "addr_state_SD",
    "addr_state_TN",
    "addr_state_TX",
    "addr_state_UT",
    "addr_state_VA",
    "addr_state_VT",
    "addr_state_WA",
    "addr_state_WI",
    "addr_state_WV",
    "addr_state_WY",
    "grade_A",
    "grade_B",
    "grade_C",
    "grade_D",
    "grade_E",
    "grade_F",
    "grade_G",
    "credit_history_length",
]

importance_df = pd.DataFrame(
    {
        "feature": feature_names,
        "importance_mean": result.importances_mean,
        "importance_std": result.importances_std,
    }
).sort_values(by="importance_mean", ascending=False)


importance_df

                       feature  importance_mean  importance_std
2                     int_rate         0.101673        0.000663
6               inq_last_6mths         0.068062        0.000709
21  purpose_debt_consolidation         0.061659        0.000357
76                     grade_B         0.054075        0.000170
75                     grade_A         0.050696        0.000393
..                         ...              ...             ...
65               addr_state_SD         0.000487        0.000029
45               addr_state_ME         0.000345        0.000017
52               addr_state_ND         0.000309        0.000034
37               addr_state_ID         0.000191        0.000027
36               addr_state_IA         0.000005        0.000004

[83 rows x 3 columns]


In [ ]:
import shap

explainer = shap.Explainer(model.predict, X_test)
shap_values = explainer(X_test)

shap.summary_plot(shap_values, X_test)

PermutationExplainer explainer:   9%|▊         | 35042/407339 [21:15<3:47:28, 27.28it/s]